# 📥 Data Import in pandas — a hands-on practice workbook
**Domain:** credit-risk data (credit cards & personal loans) · **Goal:** understand *why* each import option exists, not memorise syntax.

### How to use this notebook (the learning protocol)
| Step | What you do | Why it works |
|---|---|---|
| 🔮 **Predict** | Before running a cell, write your prediction in the markdown cell above it (shape? dtypes? error?) | Prediction errors are what drive learning (retrieval + surprise) |
| ▶️ **Run** | Run it, compare with your prediction | Immediate feedback |
| 🔧 **Break it** | Change one argument and re-run | Builds a causal model of each parameter |
| ✍️ **Exercise** | Fill in `...` cells, then run the ✅ check cell | Generation beats re-reading |
| 🧠 **Recall** | Answer the recall questions *from memory* before opening the answers | Spaced retrieval = long-term retention |
| 🗣️ **Teach back** | Explain each section in 3 sentences, out loud | Reveals gaps in understanding |

**Rule:** don't copy-paste solutions. If you're stuck for >5 minutes, read the *error message* first — it's the best teacher.

### Datasets (all in `data/`, synthetic — no real customers)
| File | Source type | What's in it |
|---|---|---|
| `credit_accounts.csv` | clean CSV | 2,000 accounts: limit, balance, bureau score, DPD, default flag |
| `credit_accounts_messy.csv` | messy CSV | metadata header, comment line, `$` strings, `dd/mm/yyyy`, weird missing markers, footer |
| `repayments_pipe.txt` | delimited text, **no header** | 3,000 repayments |
| `bureau_report_fixed_width.txt` | fixed-width text | mainframe-style bureau extract |
| `portfolio_report.xlsx` | Excel, 4 sheets | one sheet has title rows above the header |
| `customers_flat.json`, `customers_nested.json`, `customers.jsonl` | JSON variants | 1,000 customers |
| `credit_risk.db` | SQLite (stand-in for SQL Server) | `customers`, `accounts`, `payments` tables |
| `mock_api.py` | local REST API | paginated, token-protected customer endpoint |

In [ ]:
import pandas as pd, numpy as np, json, os, io
from pathlib import Path
from IPython.display import display

DATA = Path("data")
print("pandas", pd.__version__, "| files:", sorted(p.name for p in DATA.iterdir()))

def audit(df, name="df"):
    """Post-import health check. Run this after EVERY import."""
    print(f"── {name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    display(pd.DataFrame({"dtype": df.dtypes.astype(str), "nulls": df.isna().sum(),
                          "null_%": (df.isna().mean() * 100).round(1), "unique": df.nunique()}))
    print("duplicate rows:", df.duplicated().sum())

## 1 · The mental model: every import is four decisions
```
 LOCATE  →  PARSE  →  TYPE  →  VALIDATE
 where is   how do we   what is each   did we get what
 the data?  split it    column really? we expected?
```
* **Locate** – path, URL, connection string, API endpoint, credentials.
* **Parse** – delimiter, header row, encoding, quoting, which sheet, JSON structure.
* **Type** – numbers vs text, dates, categories, missing-value markers. *This is where most silent bugs live.*
* **Validate** – row counts, null counts, duplicates, ranges (that's our `audit()` function).

Every `pd.read_*` argument you'll meet answers one of these four questions. When you see a new parameter, ask: *which decision does this control?*

> 🧠 **Credit-risk angle:** a wrongly-parsed date or a `"N/A"` read as a string can silently shift default rates or break a join key. Validation isn't optional.

## 2 · CSV files — `pd.read_csv`
### 2.1 The default read
**🔮 Predict:** (a) how many rows/cols? (b) what dtype will `open_date` get? (c) which columns will be numeric? Write it here: _..._

In [ ]:
df = pd.read_csv(DATA / "credit_accounts.csv")
display(df.head())
audit(df, "credit_accounts.csv")

**Insight:** `open_date` came in as text (`object` / `str`) — pandas never guesses dates by default. A text date can't be sorted chronologically or subtracted. *Parse decisions don't include types; you must ask for them.*

### 2.2 Controlling parse & type: the 6 parameters you'll use 90% of the time
| Parameter | Decision | Example |
|---|---|---|
| `usecols` | which columns (saves memory) | `usecols=["account_id","balance"]` |
| `dtype` | force types | `dtype={"product": "category"}` |
| `parse_dates` | text → datetime | `parse_dates=["open_date"]` |
| `index_col` | which column is the index | `index_col="account_id"` |
| `nrows` | peek at first N rows | `nrows=5` |
| `na_values` | extra "missing" markers | `na_values=["N/A", "-"]` |

In [ ]:
df2 = pd.read_csv(
    DATA / "credit_accounts.csv",
    usecols=["account_id", "product", "open_date", "credit_limit", "balance", "bureau_score", "default_flag"],
    dtype={"product": "category", "bureau_score": "int16", "default_flag": "int8"},
    parse_dates=["open_date"],
    index_col="account_id",
)
print(df2.dtypes, "\n")
print(f"memory: default read = {df.memory_usage(deep=True).sum()/1e3:,.0f} KB  →  tuned read = {df2.memory_usage(deep=True).sum()/1e3:,.0f} KB")
df2.loc[df2.open_date >= "2023-01-01"].shape

**🔧 Break it:** remove `parse_dates` and re-run the last line. What error (or wrong answer) do you get, and why? Now you can *feel* what a date type buys you.

### 2.3 Real-world mess: look at raw text *before* parsing
Never trust a file you haven't looked at. Print the raw lines first.

In [ ]:
with open(DATA / "credit_accounts_messy.csv") as f:
    raw = f.read().splitlines()
print("\n".join(raw[:9]))
print("   ...")
print("\n".join(raw[-2:]))

**🔮 Predict:** what happens with a naive `pd.read_csv(path)`? List every problem you can see in the raw lines above (aim for 6) before running the next cell.

In [ ]:
try:
    pd.read_csv(DATA / "credit_accounts_messy.csv")
except Exception as e:
    print(type(e).__name__, "→", str(e)[:200])

Now the careful version. Each argument fixes one problem:
| Problem in the file | Fix |
|---|---|
| 3 metadata lines before the header | `skiprows=3` |
| a `# ...` comment line inside the data | `comment="#"` |
| footer line `Total rows: 300` | `skipfooter=1` (needs `engine="python"`) |
| missing markers `N/A`, `-`, `?` | `na_values=["-", "?"]` (`N/A` is already a default) |
| dates are `dd/mm/yyyy` | `parse_dates=[...], date_format="%d/%m/%Y"` |
| `"$10,000"` strings | `thousands=","` is **not enough** (the `$` blocks it) → clean with `.str.replace` |

In [ ]:
m = pd.read_csv(
    DATA / "credit_accounts_messy.csv",
    skiprows=3, comment="#", skipfooter=1, engine="python",
    na_values=["-", "?"],
    parse_dates=["open_date"], date_format="%d/%m/%Y",
)
for c in ["credit_limit", "balance"]:
    m[c] = m[c].str.replace(r"[$,]", "", regex=True).astype(float)
audit(m, "messy, cleaned")
m.head(3)

> 💡 **Why `date_format` matters:** `04/10/2020` is 4 October in Australia/UK but 10 April in the US. pandas will *not* warn you — it just picks one. Always state the format for ambiguous dates.

### ✍️ Exercise 2 — Load the messy file yourself
Without looking above, produce `df_messy` with: correct header, no comment/footer rows, `open_date` as datetime, `credit_limit` & `balance` as float, and the three missing markers turned into real `NaN`.

In [ ]:
df_messy = pd.read_csv(DATA / "credit_accounts_messy.csv", skiprows=3, comment="#", skipfooter=1, engine="python",
                       na_values=["-", "?"], parse_dates=["open_date"], date_format="%d/%m/%Y")
for c in ["credit_limit", "balance"]:
    df_messy[c] = df_messy[c].str.replace(r"[$,]", "", regex=True).astype(float)

In [ ]:
assert isinstance(df_messy, pd.DataFrame), "df_messy isn't a DataFrame yet"
assert df_messy.shape == (300, 9), f"expected (300, 9), got {df_messy.shape} — footer/comment/metadata rows still in?"
assert str(df_messy.open_date.dtype).startswith("datetime"), "open_date should be datetime"
assert df_messy.credit_limit.dtype == "float64" and df_messy.balance.dtype == "float64", "clean the $ and , characters"
assert df_messy[["bureau_score", "days_past_due", "product"]].isna().sum().tolist() == [12, 12, 12], "missing markers not all converted to NaN"
print("✅ Exercise 2 passed")

### 🧠 Recall (answer from memory, then open)
1. What's the difference between `skiprows` and `header`?
2. Why does `thousands=","` fail on `"$10,000"`?
3. Why can't you use `skipfooter` with the default C engine?

<details><summary>Answers</summary>

1. `skiprows` discards lines *before parsing starts*; `header=n` says which (remaining) row holds column names. They compose: skip junk, then pick the header.
2. The parser needs the whole token to look like a number once the thousands separator is removed; `$10000` still has a `$`, so it stays text.
3. The fast C parser streams the file and can't know where "the last row" is; the Python engine can look ahead.
</details>

## 3 · Text files — delimited and fixed-width
`.txt` is just a container; the *structure inside* decides the tool. `read_csv` is really "read any delimited text" — `sep=` is the key.

### 3.1 Delimited text without a header
Look at the raw file first:

In [ ]:
print(open(DATA / "repayments_pipe.txt").read()[:200])

**🔮 Predict:** if you call `pd.read_csv(path)` with defaults, what will the column names be, and what will go wrong with the first data row? Then run the next cell.

In [ ]:
bad = pd.read_csv(DATA / "repayments_pipe.txt")
print(bad.shape, list(bad.columns)[:4], " ← first data row became the header, and sep='|' wasn't used")

In [ ]:
pay = pd.read_csv(DATA / "repayments_pipe.txt", sep="|", header=None,
                  names=["account_id", "pay_date", "amount", "channel"], parse_dates=["pay_date"])
audit(pay, "repayments")

`header=None` = "there is no header row"; `names=[...]` = "here are the column names". Other separators: `sep="\t"` (TSV), `sep=";"` (EU CSV), `sep=r"\s+"` (any whitespace).

### 3.2 Fixed-width files (mainframe / bureau extracts)
Fields are defined by **character positions**, not delimiters. Layout: `id 0–7 | name 7–27 | score 27–30 | enquiries 30–32 | status 32–40`

In [ ]:
print(open(DATA / "bureau_report_fixed_width.txt").read()[:160])
bureau = pd.read_fwf(DATA / "bureau_report_fixed_width.txt",
                     colspecs=[(0, 7), (7, 27), (27, 30), (30, 32), (32, 40)],
                     names=["account_id", "customer_name", "bureau_score", "enquiries", "status"])
audit(bureau, "bureau fixed-width")

**🔧 Break it:** change the score colspec to `(27, 29)`. What happens to the scores? This is why the layout document is a *contract* — an off-by-one silently corrupts data.

### ✍️ Exercise 3 — Total repayments by channel
Using `pay` (or re-reading the file), build `total_by_channel`: a Series of total `amount` per `channel`, sorted largest first.

In [ ]:
total_by_channel = pay.groupby("channel")["amount"].sum().sort_values(ascending=False)

In [ ]:
assert isinstance(total_by_channel, pd.Series), "total_by_channel isn't a Series yet"
assert set(total_by_channel.index) == {"BPAY", "DirectDebit", "Card", "Branch"}, "expected 4 channels"
assert total_by_channel.is_monotonic_decreasing, "sort largest → smallest"
assert abs(total_by_channel.sum() - pay.amount.sum()) < 1e-6
print("✅ Exercise 3 passed:", total_by_channel.round(0).to_dict())

### 🧠 Recall
1. A file has no header and uses `;` as separator. Write the call.
2. When do you choose `read_fwf` over `read_csv`?
3. A text file shows `ï»¿` at the start of the first column name. What's wrong and what's the fix?

<details><summary>Answers</summary>

1. `pd.read_csv(path, sep=";", header=None, names=[...])`
2. When fields are aligned by character position with no delimiter (legacy/mainframe/regulatory extracts).
3. A UTF-8 BOM read with the wrong encoding → `encoding="utf-8-sig"` (or `encoding="latin-1"` for legacy Windows files).
</details>

## 4 · Excel — `pd.read_excel`
Excel is a **container of sheets**, often with decoration (titles, merged cells, notes). The key decisions: *which sheet* and *which row is the header*.

In [ ]:
xl = pd.ExcelFile(DATA / "portfolio_report.xlsx")
print("sheets:", xl.sheet_names)
raw_accounts = pd.read_excel(xl, sheet_name="Accounts", header=None, nrows=6)
raw_accounts          # ← look at the raw grid first: where is the real header?

**🔮 Predict:** the table's header is on spreadsheet row 4. What `header=` value (0-based!) selects it? Try it below, then verify the columns look right.

In [ ]:
acc_x = pd.read_excel(xl, sheet_name="Accounts", header=3)
print(acc_x.shape); display(acc_x.head(3))

# All sheets at once → dict of DataFrames
sheets = pd.read_excel(xl, sheet_name=None)
{name: df_.shape for name, df_ in sheets.items()}

Useful extras: `usecols="A:F"` (Excel column letters), `skiprows`, `nrows`, `dtype`, `na_values`, `converters`. Dates stored as real Excel dates arrive as `datetime` automatically (unlike CSV).

> ⚠️ Excel pitfalls: IDs like `00123` lose leading zeros (force `dtype=str`), formulas are read as their *cached values*, merged header cells produce `Unnamed: n` columns, and `.xls` (old) needs `xlrd`.

### ✍️ Exercise 4 — Accounts + Collections
Load the **Accounts** sheet (careful with the header) and the **Collections** sheet, then left-join collections onto accounts by `account_id` into `acc_coll`.

In [ ]:
accounts_x = pd.read_excel(DATA / "portfolio_report.xlsx", sheet_name="Accounts", header=3)
collections_x = pd.read_excel(DATA / "portfolio_report.xlsx", sheet_name="Collections")
acc_coll = accounts_x.merge(collections_x, on="account_id", how="left")

In [ ]:
assert all(isinstance(x, pd.DataFrame) for x in (accounts_x, collections_x, acc_coll)), "fill in all three DataFrames"
assert accounts_x.shape == (500, 10), f"Accounts shape {accounts_x.shape} — is header=3?"
assert "amount_recovered" in acc_coll.columns and len(acc_coll) == len(accounts_x), "left join should keep all 500 accounts"
print("✅ Exercise 4 passed — accounts in collections:", acc_coll.amount_recovered.notna().sum())

### 🧠 Recall
1. How do you get every sheet in one call?
2. Header is on row 6 of the sheet. What's `header=`?
3. Why is `pd.ExcelFile` useful when reading several sheets from one big file?

<details><summary>Answers</summary>

1. `sheet_name=None` → dict `{sheet: DataFrame}`.
2. `header=5` (0-based).
3. It opens/parses the workbook once, so each `read_excel(xl, sheet_name=...)` doesn't re-load the whole file.
</details>

## 5 · JSON — flat, nested and line-delimited
JSON is **hierarchical**; DataFrames are **rectangular**. The skill is *flattening*.
### 5.1 Flat records

In [ ]:
cust_flat = pd.read_json(DATA / "customers_flat.json")
audit(cust_flat, "customers_flat.json")

### 5.2 Nested JSON — look at the structure first

In [ ]:
with open(DATA / "customers_nested.json") as f:
    nested = json.load(f)
print(json.dumps(nested[0], indent=2))

**🔮 Predict:** `pd.DataFrame(nested)` — what will the `profile` and `contacts` columns contain? Then compare with `json_normalize`.

In [ ]:
print(pd.DataFrame(nested).iloc[0][["profile", "contacts"]].to_dict(), "\n")   # dicts/lists stuffed in cells = not analysable

flat = pd.json_normalize(nested, sep="_")        # nested dicts → columns (profile_region ...)
print(flat.columns.tolist())

contacts = pd.json_normalize(nested, record_path="contacts", meta=["customer_id"])   # list → one row per item
display(contacts.head())

* `json_normalize(..., sep="_")` flattens **dicts** into columns.
* `record_path="contacts"` explodes a **list** into rows; `meta=[...]` carries parent keys down so you can join back.

### 5.3 JSON Lines (`.jsonl`) — one record per line (logs, streaming exports, big files)

In [ ]:
jl = pd.read_json(DATA / "customers.jsonl", lines=True)       # add chunksize=... for huge files
jl.shape

### ✍️ Exercise 5 — Email verification rate
Using `contacts`, compute `email_verified_rate`: the share of customers whose **email** contact is verified (a float between 0 and 1).

In [ ]:
email_verified_rate = contacts.loc[contacts["type"] == "email", "verified"].mean()

In [ ]:
assert isinstance(email_verified_rate, (float, np.floating)), "email_verified_rate should be a single number"
assert 0.7 < email_verified_rate < 0.9, f"got {email_verified_rate:.3f} — filter to type == 'email' and take the mean of verified"
print(f"✅ Exercise 5 passed: {email_verified_rate:.1%}")

### 🧠 Recall
1. Dict inside a record vs list inside a record — which `json_normalize` feature handles each?
2. Why prefer JSON Lines for very large exports?

<details><summary>Answers</summary>

1. Dict → automatic flattening with `sep`; list → `record_path` (+ `meta` to keep parent keys).
2. Each line is independent, so you can stream/chunk it without loading one giant array into memory.
</details>

## 6 · SQL databases — SQLAlchemy + `pd.read_sql`
**Core idea:** a *connection string* (where + who) → an *engine* → `pd.read_sql(query, engine)`.
We use **SQLite** because it needs no server; the pandas code is identical for SQL Server, Azure SQL, PostgreSQL… only the connection string changes.

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(f"sqlite:///{(DATA / 'credit_risk.db').as_posix()}")

tables = pd.read_sql(text("SELECT name FROM sqlite_master WHERE type='table'"), engine)
print(tables.name.tolist())

pd.read_sql(text("SELECT * FROM accounts LIMIT 5"), engine)

### 6.1 Push work to the database
**🔮 Predict:** two ways to get the Credit Card defaults — (A) `SELECT *` everything then filter in pandas, (B) filter in SQL. Same answer; which scales to a 500-million-row table and why?

In [ ]:
# A: pull everything, filter locally
a = pd.read_sql(text("SELECT * FROM accounts"), engine)
a = a[(a["product"] == "Credit Card") & (a["default_flag"] == 1)]

# B: let the database filter (and only ship the rows/columns you need)
b = pd.read_sql(
    text("SELECT account_id, balance, bureau_score FROM accounts WHERE product = :p AND default_flag = 1"),
    engine, params={"p": "Credit Card"},
)
print(len(a), len(b))

**B** — the database has indexes and does the filtering where the data lives; you transfer less over the network.
> 🔒 **Never build SQL with f-strings from user input** (`f"... WHERE x = '{user}'"`) → SQL injection. Use bound parameters (`:p` + `params=`), as above.

### 6.2 Dates, joins and chunking

In [ ]:
q = text("""
    SELECT a.account_id, a.open_date, a.product, a.default_flag, c.region, c.annual_income
    FROM accounts a JOIN customers c ON a.customer_id = c.customer_id
""")
j = pd.read_sql(q, engine, parse_dates=["open_date"])      # SQLite stores dates as text → ask for datetime
print(j.shape, j.open_date.dtype)

# Chunked read: process a huge table in slices without loading it all
total = 0
for chunk in pd.read_sql(text("SELECT amount FROM payments"), engine, chunksize=1000):
    total += chunk["amount"].sum()
print(f"total repayments (chunked): {total:,.2f}")

### 6.3 Writing back
`df.to_sql("table", engine, if_exists="replace" | "append" | "fail", index=False)`. We'll use a throw-away in-memory DB:

In [ ]:
scratch = create_engine("sqlite://")      # in-memory
j.groupby("region")["default_flag"].mean().rename("default_rate").reset_index().to_sql("region_summary", scratch, index=False)
pd.read_sql(text("SELECT * FROM region_summary"), scratch)

### ✍️ Exercise 6 — Default rate by product × region (do the aggregation **in SQL**)
Write a query joining `accounts` and `customers` and return columns `product, region, n_accounts, default_rate` — one row per product-region pair.

In [ ]:
query = """
SELECT a.product, c.region, COUNT(*) AS n_accounts, AVG(a.default_flag) AS default_rate
FROM accounts a JOIN customers c ON a.customer_id = c.customer_id
GROUP BY a.product, c.region
"""
df_sql = pd.read_sql(text(query), engine)

In [ ]:
assert df_sql.shape == (8, 4), f"expected 8 rows × 4 cols, got {df_sql.shape}"
assert set(df_sql.columns) == {"product", "region", "n_accounts", "default_rate"}
assert df_sql.n_accounts.sum() == 2000, "every account should appear exactly once"
print("✅ Exercise 6 passed"); display(df_sql.sort_values("default_rate", ascending=False).head(3))

### 6.4 Connecting to **SQL Server** and **Azure** (templates — they won't run here; read, then adapt)
Install once: `pip install pyodbc sqlalchemy` + the **Microsoft ODBC Driver 18 for SQL Server**.

**On-premises SQL Server (Windows authentication):**
```python
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

url = URL.create(
    "mssql+pyodbc",
    query={
        "odbc_connect": (
            "DRIVER={ODBC Driver 18 for SQL Server};"
            "SERVER=MY_SERVER\\INSTANCE;DATABASE=CreditRisk;"
            "Trusted_Connection=yes;TrustServerCertificate=yes;"
        )
    },
)
engine = create_engine(url)
df = pd.read_sql(text("SELECT TOP 1000 * FROM dbo.Accounts"), engine)   # T-SQL: TOP, not LIMIT
```

**SQL authentication (username/password) — read secrets from environment variables, never hard-code:**
```python
import os
odbc = (f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={os.environ['SQL_SERVER']};"
        f"DATABASE={os.environ['SQL_DB']};UID={os.environ['SQL_USER']};PWD={os.environ['SQL_PWD']};")
engine = create_engine(URL.create("mssql+pyodbc", query={"odbc_connect": odbc}))
```

**Azure SQL Database** — same pattern; the server is `yourserver.database.windows.net`, encryption is mandatory, and you can use Entra ID (Azure AD) instead of a password:
```python
odbc = ("DRIVER={ODBC Driver 18 for SQL Server};SERVER=yourserver.database.windows.net;DATABASE=CreditRisk;"
        "Authentication=ActiveDirectoryInteractive;UID=you@company.com;Encrypt=yes;")
# other options: ActiveDirectoryPassword, ActiveDirectoryServicePrincipal, ActiveDirectoryDefault
```

**Azure Blob Storage / Data Lake Gen2** (files in the cloud):
```python
# pip install adlfs azure-identity
df = pd.read_csv("abfs://container/path/credit_accounts.csv",
                 storage_options={"account_name": "mystorageacct", "anon": False})   # uses your Azure login
# or with a connection string:  storage_options={"connection_string": os.environ["AZURE_STORAGE_CONNECTION_STRING"]}

# Alternative without adlfs:
# from azure.storage.blob import BlobClient
# data = BlobClient.from_connection_string(conn_str, "container", "file.csv").download_blob().readall()
# df = pd.read_csv(io.BytesIO(data))
```
**Useful habit:** keep the *query* in one place and the *connection* in another, so the same notebook runs against dev (SQLite) and prod (SQL Server) by swapping the engine.

### 🧠 Recall
1. What are the three things a connection string answers?
2. Why `params={...}` instead of an f-string?
3. SQLite returned dates as text — what's the pandas-side fix?

<details><summary>Answers</summary>

1. Which driver/dialect, which server/database, and how to authenticate.
2. Bound parameters are sent separately from the SQL text → no injection, correct quoting/types.
3. `parse_dates=[...]` in `read_sql` (or `pd.to_datetime` afterwards).
</details>

## 7 · REST APIs — `requests` → JSON → DataFrame
The pattern is always: **request → check status → `.json()` → pick the records list → `DataFrame`**.
No internet needed: we start a tiny local API (`mock_api.py`) that behaves like a real one — token auth and pagination.

In [ ]:
import requests
from mock_api import start_server, TOKEN
start_server(8765)

BASE = "http://127.0.0.1:8765/api/v1"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

# 1) No token → what happens?
r = requests.get(f"{BASE}/customers", timeout=10)
print("without token:", r.status_code, r.json())

# 2) With token
r = requests.get(f"{BASE}/customers", headers=HEADERS, params={"page": 1, "page_size": 3}, timeout=10)
print("with token   :", r.status_code)
payload = r.json()
print({k: v for k, v in payload.items() if k != "data"})      # metadata: pagination info
pd.DataFrame(payload["data"])

Things to notice: the useful records are inside `payload["data"]`, **not** at the top level; the response also says `total_pages` — that's your loop's stop condition.

| HTTP status | Meaning | Typical reaction |
|---|---|---|
| 200 | OK | parse |
| 401/403 | bad/missing credentials | fix token, don't retry blindly |
| 404 | wrong endpoint | check URL |
| 429 | rate-limited | wait + retry (back-off) |
| 5xx | server problem | retry a few times, then fail loudly |

### ✍️ Exercise 7 — Fetch *all* pages
Loop through every page (`page_size=200`), collect records and build `df_api`. Always use `timeout=` and `raise_for_status()`.

In [ ]:
rows = []
for page in range(1, 50):
    resp = requests.get(f"{BASE}/customers", headers=HEADERS, params={"page": page, "page_size": 200}, timeout=10)
    resp.raise_for_status()
    body = resp.json()
    rows.extend(body["data"])
    if page >= body["total_pages"]:
        break
df_api = pd.DataFrame(rows)

In [ ]:
assert len(df_api) == 1000, f"expected 1000 customers, got {len(df_api)} — did you stop too early or loop forever?"
assert df_api.customer_id.is_unique, "duplicate customers — are you re-fetching the same page?"
print("✅ Exercise 7 passed"); audit(df_api, "customers via API")

### 7.1 Calling a real API (template)
```python
resp = requests.get(
    "https://api.example.com/v1/rates",
    headers={"Authorization": f"Bearer {os.environ['API_TOKEN']}"},   # keys live in env vars, not in notebooks!
    params={"from": "2026-01-01", "to": "2026-09-30"},
    timeout=15,
)
resp.raise_for_status()
df = pd.json_normalize(resp.json()["results"])
```
Production habits: retries with back-off (`urllib3.util.Retry` / `tenacity`), respect rate limits, log the request params, and never commit tokens to Git.

### 🧠 Recall
1. Why `timeout=`? 2. Where do API keys belong? 3. Your DataFrame has one column full of dicts — which function do you reach for?

<details><summary>Answers</summary>

1. Without it, `requests` can wait forever on a hung server.
2. Environment variables / secret manager — never in the notebook or Git.
3. `pd.json_normalize` (dicts → columns, lists → `record_path`).
</details>

## 8 · Other sources worth knowing
| Source | Call | When |
|---|---|---|
| **Parquet** | `pd.read_parquet` / `df.to_parquet` | big analytic data: typed, compressed, fast |
| **SAS / Stata / SPSS** | `pd.read_sas`, `pd.read_stata`, `pd.read_spss` | legacy risk models often live in `.sas7bdat` |
| **HTML tables** | `pd.read_html` | tables on web pages (needs `lxml`) |
| **Clipboard** | `pd.read_clipboard()` | quick paste from Excel |
| **Pickle** | `pd.read_pickle` | Python-only snapshots (never load untrusted pickles!) |

In [ ]:
# Parquet: types survive the round trip — CSV loses them
accounts = pd.read_csv(DATA / "credit_accounts.csv", parse_dates=["open_date"])
accounts.to_parquet("accounts_tmp.parquet")
back = pd.read_parquet("accounts_tmp.parquet")
print("dtypes preserved:", back.dtypes.equals(accounts.dtypes),
      "| csv KB:", round((DATA / "credit_accounts.csv").stat().st_size / 1e3), "| parquet KB:", round(Path("accounts_tmp.parquet").stat().st_size / 1e3))
os.remove("accounts_tmp.parquet")

# HTML table parsed straight from a string
html = "<table><tr><th>band</th><th>pd_pct</th></tr><tr><td>A</td><td>0.4</td></tr><tr><td>B</td><td>1.8</td></tr></table>"
pd.read_html(io.StringIO(html))[0]

## 9 · Capstone — build an analytic base table from **four** sources
Real projects join data from several systems. You'll combine:
* accounts → **CSV** · customers → **SQL** · collections → **Excel** · repayments → **pipe text file**

Notice the two safety nets: `validate="m:1"` (fails if the "one" side isn't unique) and a row-count assertion after joins — the #1 cause of silent double-counting.

In [ ]:
accounts    = pd.read_csv(DATA / "credit_accounts.csv", parse_dates=["open_date"])
customers   = pd.read_sql(text("SELECT * FROM customers"), engine, parse_dates=["customer_since"])
collections = pd.read_excel(DATA / "portfolio_report.xlsx", sheet_name="Collections")
payments    = pd.read_csv(DATA / "repayments_pipe.txt", sep="|", header=None,
                          names=["account_id", "pay_date", "amount", "channel"], parse_dates=["pay_date"])

pay_agg = payments.groupby("account_id").agg(n_payments=("amount", "size"), total_paid=("amount", "sum")).reset_index()

abt = (accounts
       .merge(customers,   on="customer_id", how="left", validate="m:1")
       .merge(pay_agg,     on="account_id",  how="left", validate="1:1")
       .merge(collections, on="account_id",  how="left", validate="1:1"))

assert len(abt) == len(accounts), "a join changed the row count!"
audit(abt, "analytic base table")

**🔮 Predict:** will the default rate *rise or fall* as the bureau score band improves? By how much between the worst and best band? Then run it.

In [ ]:
bands = pd.cut(abt.bureau_score, bins=[299, 579, 669, 739, 799, 900],
               labels=["Poor", "Fair", "Good", "Very good", "Excellent"])
(abt.groupby(bands, observed=True)["default_flag"].agg(accounts="size", default_rate="mean").round(3))

### ✍️ Exercise 8 (capstone) — your own question
1. Create `util_bucket` with `pd.cut` on `utilisation` (e.g. `[0, .25, .5, .75, 1.0]`, `include_lowest=True`).
2. Build `dr_table`: default rate and account count by `util_bucket`.
3. In the markdown cell after the check, write **one sentence** a credit-risk manager could act on. (If the data surprises you, say why.)

In [ ]:
abt["util_bucket"] = pd.cut(abt.utilisation, bins=[0, .25, .5, .75, 1.0], include_lowest=True)
dr_table = abt.groupby("util_bucket", observed=True)["default_flag"].agg(accounts="size", default_rate="mean")

In [ ]:
assert isinstance(dr_table, pd.DataFrame), "build dr_table first"
assert "util_bucket" in abt.columns and abt.util_bucket.notna().all(), "every account should land in a bucket"
assert dr_table.accounts.sum() == 2000
print("✅ Exercise 8 passed"); display(dr_table.round(3))

**My insight (write here):** _..._

## 10 · Consolidation
### Decision cheat-sheet — "which tool for which source?"
| I have… | First look at… | Call | Watch out for |
|---|---|---|---|
| clean CSV | `head` of raw text | `read_csv` | date & dtype inference |
| messy CSV / report export | raw lines | `read_csv(skiprows, comment, skipfooter, na_values)` | currency text, ambiguous dates |
| `.txt` with a delimiter | raw text | `read_csv(sep=...)` | no header → `header=None, names=` |
| fixed-width `.txt` | layout spec | `read_fwf(colspecs=...)` | off-by-one positions |
| Excel | `ExcelFile.sheet_names`, raw grid | `read_excel(sheet_name, header)` | title rows, lost leading zeros |
| JSON (nested) | `json.dumps(obj[0], indent=2)` | `json_normalize(record_path, meta)` | lists inside cells |
| SQL Server / Azure SQL | the schema | `read_sql(text(q), engine, params)` | injection, huge tables → filter in SQL / chunk |
| REST API | docs + one sample response | `requests.get → .json() → DataFrame` | auth, pagination, rate limits, timeouts |
| Azure Blob / ADLS | storage account/container | `read_csv("abfs://…", storage_options)` | credentials, `adlfs` installed |

### The 5-line import ritual (use it every time)
1. **Look** at the raw source (text lines / sheet grid / one JSON record / `LIMIT 5`).
2. **Read** with explicit parse + type arguments.
3. **Audit** (`shape`, `dtypes`, nulls, duplicates, ranges).
4. **Assert** the things that must be true (row counts, uniqueness, date ranges).
5. **Save** a clean copy (parquet) so you never re-clean.

### 🧠 Final spaced-recall set (do this again tomorrow, in 3 days, and in a week)
1. Name the four decisions behind every import.
2. Which argument tells `read_csv` there's no header, and which names the columns?
3. How do you read a sheet whose table starts on row 8?
4. When do you use `record_path` vs plain `json_normalize`?
5. Why should filtering happen in SQL, and how do you pass values safely?
6. What do `validate="m:1"` and a row-count assertion protect you from?
7. Your API returns 50 records per page. What's the loop's exit condition?
8. CSV vs Parquet: what does each preserve?

<details><summary>Answers</summary>

1. Locate, parse, type, validate.
2. `header=None`; `names=[...]`.
3. `header=7` (0-based) — or `skiprows=7`.
4. `record_path` when a *list* inside each record should become rows; plain `json_normalize` flattens dicts.
5. The DB does the work near the data and sends fewer rows; bound parameters (`:p`, `params=`).
6. Silent row duplication/loss from joins on non-unique keys.
7. When `page >= total_pages` (or the returned `data` is empty / `next` is null).
8. CSV: text only, types are re-guessed; Parquet: dtypes, compressed, columnar.
</details>

### 🚀 Next challenges (stretch)
* Load `credit_accounts.csv` with `chunksize=500` and compute the default rate without holding the full file in memory.
* Swap the SQLite engine for your real SQL Server engine and re-run Exercise 6 unchanged.
* Write a function `load_all()` that returns the ABT and saves it to Parquet; then time CSV vs Parquet reads with `%timeit`.